# Chapter 1 — Vectors, Matrices, and Linear Maps

Companion notebook for *The Math That Powers AI* (2nd ed), Chapter 1 — all functions are imported from the repo's `mathpowersai` package (`src/mathpowersai/linear_algebra.py`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

from mathpowersai.linear_algebra import (
    TOY_EMBEDDING,
    attention_demo,
    cosine_similarity,
    dot,
    norm,
    project,
    similarity_demo,
    word_analogy,
)

## Dot product and cosine similarity

The chapter builds up from the dot product $u \cdot v = \sum_i u_i v_i$ and the $\ell_p$ norm to cosine similarity $\mathrm{sim}(u, v) = \frac{u \cdot v}{\lVert u \rVert\, \lVert v \rVert} = \cos\theta$. On the Table 1.1 toy word embedding, `similarity_demo()` reproduces the book's listing: *king* and *queen* point in nearly the same direction (similarity close to 1), while *king* and *apple* do not.

In [ ]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, -5.0, 6.0])
print(f"u . v          = {dot(u, v)}")
print(f"||u||_2        = {norm(u):.3f}")
print(f"||u||_1        = {norm(u, p=1):.3f}")
print(f"||u||_inf      = {norm(u, p=np.inf):.3f}")
print(f"sim(u, v)      = {cosine_similarity(u, v):.3f}")

# The chapter's Table 1.1 listing:
king_queen, king_apple = similarity_demo()
print(f"king-queen similarity: {king_queen:.3f}")  # High
print(f"king-apple similarity: {king_apple:.3f}")  # Low

## Projection

The vector projection $\mathrm{proj}_v(u) = \frac{u \cdot v}{v \cdot v}\, v$ extracts the component of $u$ along the direction of $v$. The book section shows that the residual $u - \mathrm{proj}_v(u)$ is orthogonal to $v$ — we verify that numerically below (the dot product of the residual with $v$ is zero up to floating-point error).

In [ ]:
u = np.array([3.0, 4.0])
v = np.array([1.0, 0.0])
p = project(u, v)
residual = u - p
print(f"proj_v(u)            = {p}")
print(f"residual u - proj    = {residual}")
print(f"residual . v         = {dot(residual, v)}  (orthogonal)")

# Projection onto a non-axis direction, deterministic random u.
rng = np.random.default_rng(42)
u2 = rng.standard_normal(3)
v2 = np.array([1.0, 1.0, 1.0])
p2 = project(u2, v2)
print(f"u2                   = {u2.round(3)}")
print(f"proj_v2(u2)          = {p2.round(3)}")
print(f"(u2 - proj) . v2     = {dot(u2 - p2, v2):.2e}")

## Simplified self-attention

The chapter's self-attention listing shows how the dot product powers transformers: for word embeddings $v_1, \dots, v_n$, the scores $\mathrm{score}_{ij} = v_i \cdot v_j$ measure how relevant word $j$ is to word $i$; a row-wise softmax turns the scores into weights $\alpha_{ij}$; and each new embedding is the linear combination $v'_i = \sum_j \alpha_{ij} v_j$. `attention_demo()` runs this over the three 4-dimensional embeddings for "the", "cat", "sat".

In [ ]:
V, scores, weights, V_new = attention_demo()
print("Word embeddings V (rows: 'the', 'cat', 'sat'):")
print(V)
print("\nAttention scores (V @ V.T):")
print(scores)
print("\nAttention weights (row-wise softmax):")
print(weights.round(3))
print("\nRow sums of weights:", weights.sum(axis=1).round(3))
print("\nNew embeddings (weights @ V):")
print(V_new.round(3))

## Word analogies

The chapter closes with the word analogy task: in a good embedding space, vector arithmetic captures meaning, so $v_{\text{king}} - v_{\text{man}} + v_{\text{woman}} \approx v_{\text{queen}}$. `word_analogy(a, b, c)` computes the target $v_a - v_b + v_c$ and returns the vocabulary word (excluding $a$, $b$, $c$) with the highest cosine similarity to it, using the Table 1.1 toy embedding.

In [ ]:
print("Vocabulary:", sorted(TOY_EMBEDDING))

answer = word_analogy("king", "man", "woman")
print(f"king - man + woman  ~= {answer}")

answer2 = word_analogy("queen", "woman", "man")
print(f"queen - woman + man ~= {answer2}")

# Show why: similarity of every candidate to the target vector.
target = (TOY_EMBEDDING["king"] - TOY_EMBEDDING["man"]
          + TOY_EMBEDDING["woman"])
for w in sorted(TOY_EMBEDDING):
    if w in ("king", "man", "woman"):
        continue
    sim = cosine_similarity(target, TOY_EMBEDDING[w])
    print(f"  sim(target, {w:<6s}) = {sim:.3f}")